# [PBT] 화장품 이커머스 ㅣ 08. 마스터 테이블

## 이 노트북이 하는 일

04번의 설명변수(X)와 05번의 목표변수(Y)를 **하나의 분석 테이블로 합친다.**
여기까지가 계획서 1일차 산출물이며, 09번 EDA부터 두 사람이 갈라진다.

| 입력 | 내용 |
|---|---|
| `02b_x.csv` | 108,458행 — 설명변수 6종 + 기본 6열 |
| `02c_y.csv` | 108,458행 — `purchase_7d` · `lag_hours` + 기본 6열 |

| 출력 | 내용 |
|---|---|
| `03_master.csv` | 108,458행 — 1행 = 1고객. X · Y · 식별자 · 대조용 변수 |

## 이 노트북에서 새로 하는 일은 두 가지뿐이다

**① 결합** — 두 파일이 기본 6열(`user_id`·`t0`·`product_id`·`price`·`brand`·`category_id`)을
중복해서 갖고 있으므로, Y 쪽에서는 필요한 3열만 가져온다.

**② 범주 묶기** — 06번에서 확정한 기준을 실제 값으로 적용한다.

```
brand_grp    = brand 결측을 '미상'으로 채운 뒤, 1,000건 미만을 '기타'로  →  X 포함
category_grp = category_id 를 500건 미만은 '기타'로                    →  X 제외, 대조용
```

> ⚠️ **더미변수는 만들지 않는다.** 범주형 값 그대로 담는다.
> 더미 인코딩은 알고리즘별 `drop_first` 옵션이 달라 **모델링 단계에서** 수행한다.
> 
> 더미 인코딩·다중공선성 확인·스케일링은 **분할 후** 각각 수행한다.

## `category_grp`를 왜 담는가

06번에서 **X에서 제외**하기로 확정했다. 그런데도 컬럼으로 담아두는 이유는
4일차에 **민감도 검증**을 해야 하기 때문이다.

품목을 빼면 교란 위험이 남는다. 고관여 상품은 오래 보고 덜 사고, 저가 소모품은 덜 보고 더 살 수 있다.
그러면 "조회를 많이 하면 덜 산다"는 **가짜 결론**이 나오는데, 그것이 이 프로젝트의 핵심 질문이다.

그래서 11번에서 X 목록만 바꿔 두 번 적합하고, 핵심 X의 결론이 흔들리는지 대조한다.
**컬럼은 여기 있되, X 목록에는 넣지 않는다.** `lag_hours`와 같은 취급이다.

## #01. 준비작업

### 1. 라이브러리 참조

In [ ]:
import os, sys

# 프로젝트 폴더 — notebooks/ 안에서 실행하면 상위 폴더를 가리킨다
from pathlib import Path
BASE = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
os.chdir(BASE)             # 폰트 경로가 상대경로라 여기서 실행해야 한다
sys.path.insert(0, BASE)

from pandas import read_csv, read_parquet, to_datetime
from pandas import DataFrame, Timedelta, set_option

from helpers import my_qtcheck

set_option("display.max_columns", 50)

BRAND_MIN = 1000    # 데이터의 자연 절단면
CAT_MIN   =  500    # 민감도 대조용, 최대 통제
WINDOW    =   30    # 조회 변수 구간(분). 수기 대조에 쓴다

### 2. 경로 지정

In [37]:

OUT = BASE + r"\output"

## #02. 불러오기

CSV는 타입을 보존하지 않는다. 01번에서 정한 타입을 여기서 되살린다.

| 컬럼 | 되살릴 타입 | 근거 |
|---|---|---|
| `t0` | datetime | 시간 계산을 한다 |
| `product_id` | int32 | 01번 `#03-6`에서 정한 타입 |
| `brand` · `category_id` | **문자열로 둔다** | 범주를 새로 묶어야 한다. category 타입은 없는 값('기타')을 넣을 수 없다 |

In [38]:
x = read_csv(OUT + r"\02b_x.csv", encoding="utf-8-sig")
y = read_csv(OUT + r"\02c_y.csv", encoding="utf-8-sig")

print(f"X : {len(x):,}행 × {x.shape[1]}열")
print(f"Y : {len(y):,}행 × {y.shape[1]}열")
print()
print("X 컬럼 :", list(x.columns))
print("Y 컬럼 :", list(y.columns))

X : 108,458행 × 12열
Y : 108,458행 × 8열

X 컬럼 : ['user_id', 't0', 'product_id', 'price', 'brand', 'category_id', 'prior_view_cnt', 'prior_uniq_product_cnt', 'cart_product_view_cnt', 'has_prior_view', 'cart_timeband', 'cart_weekday']
Y 컬럼 : ['user_id', 't0', 'product_id', 'price', 'brand', 'category_id', 'purchase_7d', 'lag_hours']


## #03. 결합

### 1. Y에서 3열만 가져오는 이유

두 파일이 **기본 6열을 중복해서** 갖고 있다.
그대로 병합하면 `price_x` · `price_y` 처럼 접미사가 붙은 열이 6쌍 생긴다.

Y 쪽에서 필요한 것은 **`user_id`(키) · `purchase_7d` · `lag_hours`** 셋뿐이다.

> 07번 번인 진단에서 쓴 방식과 동일하다.

In [39]:
m = x.merge(y[["user_id", "purchase_7d", "lag_hours"]], on="user_id", how="inner")

print(f"결합 결과 : {len(m):,}행 × {m.shape[1]}열")
print(f"X 행 수와 일치 : {len(m) == len(x)}")
print(f"user_id 중복   : {m['user_id'].duplicated().sum()}")
print(f"purchase_7d 결측 : {m['purchase_7d'].isna().sum()}")

결합 결과 : 108,458행 × 14열
X 행 수와 일치 : True
user_id 중복   : 0
purchase_7d 결측 : 0


### 2. 타입 복원

- `cart_timeband`는 **순서가 있는** 범주다(심야 → 오전 → 오후 → 저녁). 순서를 지정해 둔다.
- `lag_hours`의 결측은 **정상이다.** "7일 내에 사지 않았다"는 뜻이다.

In [40]:
m["t0"]         = to_datetime(m["t0"], format="%Y-%m-%d %H:%M:%S")
m["product_id"] = m["product_id"].astype("int32")

for c in ["prior_view_cnt", "prior_uniq_product_cnt", "cart_product_view_cnt"]:
    m[c] = m[c].astype("int32")

m["has_prior_view"] = m["has_prior_view"].astype("int8")
m["purchase_7d"]    = m["purchase_7d"].astype("int8")

m["cart_timeband"] = m["cart_timeband"].astype("category").cat.set_categories(
    ["심야", "오전", "오후", "저녁"], ordered=True)
m["cart_weekday"]  = m["cart_weekday"].astype("category").cat.set_categories(
    ["월", "화", "수", "목", "금", "토", "일"], ordered=True)

m.dtypes.to_frame("dtype")

,dtype
user_id,int64
t0,datetime64[us]
product_id,int32
price,float64
brand,str
category_id,int64
prior_view_cnt,int32
prior_uniq_product_cnt,int32
cart_product_view_cnt,int32
has_prior_view,int8


## #04. 범주 묶기 — 06번 결정 적용

### 1. 묶기 함수

06번 `#05-1`과 **같은 함수**다. 기준을 넘는 범주만 남기고 나머지를 '기타'로 바꾼다.
"상위 N개를 고른다"가 아니라 **기준을 넘는 것을 남기므로 개수는 결과적으로 정해진다.**

> ⚠️ **순서가 중요하다.** `brand` 결측을 '미상'으로 **먼저** 채운다.
> 결측 상태로 건수를 세면 그 38.85%가 집계에서 통째로 빠져,
> 가장 큰 범주가 사라진 채로 묶이게 된다.

In [41]:
def group_rare(s, min_count, other="기타"):
    counts = s.value_counts()
    keep = counts[counts >= min_count].index
    return s.where(s.isin(keep), other)


m["brand_grp"]    = group_rare(m["brand"].astype("string").fillna("미상"), BRAND_MIN)
m["category_grp"] = group_rare(m["category_id"].astype("string"),          CAT_MIN)

m["brand_grp"]    = m["brand_grp"].astype("category")
m["category_grp"] = m["category_grp"].astype("category")

print(f"brand_grp    : {m['brand_grp'].nunique():>3}종   (기준 {BRAND_MIN:,}건)")
print(f"category_grp : {m['category_grp'].nunique():>3}종   (기준 {CAT_MIN:,}건)")

brand_grp    :  15종   (기준 1,000건)
category_grp :  59종   (기준 500건)


### 2. 06번 결과와 일치하는가

06번은 같은 108,458행 테이블(`02c_y.csv`)에서 기준을 정했으므로 **종수가 정확히 같아야 한다.**
다르면 둘 중 하나가 잘못된 것이다.

In [42]:
chk = DataFrame({
    "06번 확정": [15, 59],
    "여기 결과": [m["brand_grp"].nunique(), m["category_grp"].nunique()],
}, index=["brand_grp", "category_grp"])

chk["일치"]      = chk["06번 확정"] == chk["여기 결과"]
chk["더미 열"]   = chk["여기 결과"] - 1
chk["'기타' %"]  = [round((m["brand_grp"]    == "기타").mean() * 100, 1),
                    round((m["category_grp"] == "기타").mean() * 100, 1)]
chk["결측"]      = [m["brand_grp"].isna().sum(), m["category_grp"].isna().sum()]
chk

,06번 확정,여기 결과,일치,더미 열,'기타' %,결측
brand_grp,15,15,True,14,23.6,0
category_grp,59,59,True,58,32.7,0


### 3. `brand_grp` 분포

'미상'과 '기타'는 실제 브랜드가 아니라 **처리 결과로 생긴 범주**다.
크기를 알고 있어야 나중에 계수를 해석할 때 헷갈리지 않는다.

In [43]:
b_tbl = m["brand_grp"].value_counts().rename("고객 수").to_frame()
b_tbl["비율 %"] = (b_tbl["고객 수"] / len(m) * 100).round(2)
b_tbl

,고객 수,비율 %
brand_grp,,
미상,42136,38.85
기타,25627,23.63
runail,6744,6.22
irisk,5457,5.03
grattol,4741,4.37
estel,3144,2.90
masura,3062,2.82
kapous,2764,2.55
italwax,2713,2.50


## #05. 변수 역할 확정

### ⚠️ 테이블에 있다고 전부 X가 아니다

이 표가 이 노트북의 **핵심 산출물**이다. 09번 이후 모든 단계가 이 표를 따른다.

| 역할 | 변수 | 모델 투입 |
|---|---|---|
| 식별 | `user_id` · `t0` · `product_id` | ✗ |
| **목표** | `purchase_7d` | Y |
| **누수** | `lag_hours` | ✗ **절대 금지** |
| X 연속 | `price` · `prior_view_cnt` · `prior_uniq_product_cnt` · `cart_product_view_cnt` | ○ |
| X 이진 | `has_prior_view` | ○ |
| X 명목 | `cart_timeband` · `cart_weekday` · `brand_grp` | ○ |
| 대조용 | `category_grp` | ✗ (4일차 민감도에서만) |
| 추적용 | `brand` · `category_id` | ✗ (묶기 전 원본) |

**`lag_hours`는 t₀ 이후 정보이고 Y=1일 때만 값이 있다.** X에 넣는 순간 정답을 보고 맞히는 것이 된다.

**`brand` · `category_id` 원본을 모델에 넣지 않는 이유** :
> 파생변수와 원본변수를 함께 모델링에 투입하면 정보가 중복되어 모델 성능이 지나치게 높게 나온다.
> 
> 원본과 파생 중 효과 강도가 더 높은 변수만 채택해야 한다.

추적과 수기 대조를 위해 컬럼은 남기되, X 목록에는 넣지 않는다.

In [44]:
ROLE = {
    "user_id"                : ("식별",    "고객 식별자"),
    "t0"                     : ("식별",    "최초 담기 시각 (파생 재료)"),
    "product_id"             : ("식별",    "담은 상품"),
    "purchase_7d"            : ("목표 Y",  "t0 이후 7일 내 그 상품 구매 여부"),
    "lag_hours"              : ("제외",    "t0 이후 정보 — 누수"),
    "price"                  : ("X 연속",  "t0 시점 가격"),
    "prior_view_cnt"         : ("X 연속",  "t0 직전 30분 조회 건수"),
    "prior_uniq_product_cnt" : ("X 연속",  "t0 직전 30분 조회 상품 종류 수"),
    "cart_product_view_cnt"  : ("X 연속",  "t0 직전 30분 담은 상품 조회 횟수"),
    "has_prior_view"         : ("X 이진",  "t0 직전 30분 조회 여부"),
    "cart_timeband"          : ("X 명목",  "담은 시간대 4구간"),
    "cart_weekday"           : ("X 명목",  "담은 요일"),
    "brand_grp"              : ("X 명목",  "브랜드 15종 (1,000건 기준)"),
    "category_grp"           : ("제외",    "품목 59종 (500건 기준) — 4일차 민감도 대조용"),
    "brand"                  : ("추적",    "원본. brand_grp의 재료"),
    "category_id"            : ("추적",    "원본. category_grp의 재료"),
}

master = m[list(ROLE)].copy()          # 역할표 순서대로 컬럼 정렬

role = DataFrame(
    [(k, v[0], v[1], str(master[k].dtype)) for k, v in ROLE.items()],
    columns=["변수", "역할", "설명", "dtype"],
).set_index("변수")

X_COLS = [k for k, v in ROLE.items() if v[0].startswith("X ")]
print("모델 X :", len(X_COLS), "개 →", X_COLS)
print()
role

모델 X : 8 개 → ['price', 'prior_view_cnt', 'prior_uniq_product_cnt', 'cart_product_view_cnt', 'has_prior_view', 'cart_timeband', 'cart_weekday', 'brand_grp']



,역할,설명,dtype
변수,,,
user_id,식별,고객 식별자,int64
t0,식별,최초 담기 시각 (파생 재료),datetime64[us]
product_id,식별,담은 상품,int32
purchase_7d,목표 Y,t0 이후 7일 내 그 상품 구매 여부,int8
lag_hours,제외,t0 이후 정보 — 누수,float64
price,X 연속,t0 시점 가격,float64
prior_view_cnt,X 연속,t0 직전 30분 조회 건수,int32
prior_uniq_product_cnt,X 연속,t0 직전 30분 조회 상품 종류 수,int32
cart_product_view_cnt,X 연속,t0 직전 30분 담은 상품 조회 횟수,int32


## #06. 검증

하나라도 실패하면 09번으로 넘어가지 않는다.

### 1. 행 수 · 1행 = 1고객 · 목표변수 분포

In [45]:
print(f"행 수      : {len(master):,}")
print(f"고객 수    : {master['user_id'].nunique():,}")
print(f"1행=1고객  : {len(master) == master['user_id'].nunique()}")
print(f"X 행 수 유지: {len(master) == len(x)}")
print()

pos = master["purchase_7d"].mean()
print(f"Y 양성     : {int(master['purchase_7d'].sum()):,}명 / {len(master):,}명 = {pos:.2%}")
print(f"05번 값 20.50%와 일치 : {round(pos * 100, 2) == 20.50}")
print()
print(f"t0 범위    : {master['t0'].min()}  ~  {master['t0'].max()}")

행 수      : 108,458
고객 수    : 108,458
1행=1고객  : True
X 행 수 유지: True

Y 양성     : 22,236명 / 108,458명 = 20.50%
05번 값 20.50%와 일치 : True

t0 범위    : 2019-10-11 00:00:17  ~  2019-11-24 02:58:43


### 2. 결측

**허용되는 결측은 두 개뿐이다.**

| 컬럼 | 결측 | 의미 |
|---|---|---|
| `lag_hours` | 약 79.5% | 7일 내에 사지 않았다 (= `purchase_7d` 0인 사람 수와 정확히 같아야 한다) |
| `brand` | 38.85% | 원본의 결측. `brand_grp`에서 '미상'으로 처리됨 |

**`brand_grp` · `category_grp`에 결측이 있으면 묶기가 잘못된 것이다.**

In [46]:
miss = my_qtcheck.check_missing_values(master)

n0 = int((master["purchase_7d"] == 0).sum())
print()
print(f"lag_hours 결측 : {int(master['lag_hours'].isna().sum()):,}")
print(f"Y=0 인 고객    : {n0:,}")
print(f"두 값 일치     : {int(master['lag_hours'].isna().sum()) == n0}")
print()
print(f"brand_grp    결측 : {int(master['brand_grp'].isna().sum())}   (0이어야 정상)")
print(f"category_grp 결측 : {int(master['category_grp'].isna().sum())}   (0이어야 정상)")
print(f"X 컬럼 결측 합계  : {int(master[X_COLS].isna().sum().sum())}   (0이어야 정상)")


lag_hours 결측 : 86,222
Y=0 인 고객    : 86,222
두 값 일치     : True

brand_grp    결측 : 0   (0이어야 정상)
category_grp 결측 : 0   (0이어야 정상)
X 컬럼 결측 합계  : 0   (0이어야 정상)


### 3. 논리적으로 성립해야 하는 관계

04번·05번에서 확인한 관계가 결합 후에도 유지되는지 다시 본다.
병합 과정에서 행이 어긋나면 여기서 깨진다.

In [47]:
c1 = (master["cart_product_view_cnt"]  <= master["prior_view_cnt"]).all()
c2 = (master["prior_uniq_product_cnt"] <= master["prior_view_cnt"]).all()
c3 = ((master["prior_view_cnt"] == 0) | (master["prior_uniq_product_cnt"] >= 1)).all()
c4 = ((master["has_prior_view"] == 1) == (master["prior_view_cnt"] > 0)).all()

lag = master.loc[master["purchase_7d"] == 1, "lag_hours"]
c5 = ((lag > 0) & (lag <= 168)).all()
c6 = master.loc[master["purchase_7d"] == 0, "lag_hours"].isna().all()
c7 = (master["price"] > 0).all()

for name, ok in [("담은 상품 조회 ≤ 전체 조회", c1),
                 ("상품 종류 수  ≤ 전체 조회", c2),
                 ("조회 있으면 종류 ≥ 1     ", c3),
                 ("has_prior_view 정합      ", c4),
                 ("Y=1 의 시차가 0~168시간  ", c5),
                 ("Y=0 은 시차 없음         ", c6),
                 ("price 전부 양수          ", c7)]:
    print(f"{name} : {ok}")

print()
print("전부 통과 :", all([c1, c2, c3, c4, c5, c6, c7]))

담은 상품 조회 ≤ 전체 조회 : True
상품 종류 수  ≤ 전체 조회 : True
조회 있으면 종류 ≥ 1      : True
has_prior_view 정합       : True
Y=1 의 시차가 0~168시간   : True
Y=0 은 시차 없음          : True
price 전부 양수           : True

전부 통과 : True


### 4. 무작위 20명 수기 대조

계획서 1일차 산출물로 요구한 검증이다. 04번·05번에서는 3명씩만 했다.

**코드로 재계산하지 않는다.** 같은 논리로 다시 계산하면 같은 버그를 그대로 재현할 뿐이다.
아래 출력은 **규칙보다 넓은 범위**를 보여준다. 30분 창과 7일 창을 **직접 적용해서** 판정한다.

#### 출력 한 덩어리의 구조

```
[ 1] user_id ...   t0 ...   담은 상품 ...
     테이블의 주장 →  조회 N건 / 종류 N / 담은상품 N / has_prior_view N || Y N
  X 근거 — t0 직전 60분        ← 원본 로그. 규칙(30분)보다 넓게 보여준다
  Y 근거 — 그 상품 구매 기록 전부  ← 원본 로그. 기간 제한 없이 보여준다
```

**할 일은 하나다. 아래 증거를 손으로 세서 위 주장과 같은지 본다.**

#### 세는 규칙

| 확인할 것 | 세는 방법 |
|---|---|
| `prior_view_cnt` | `30분창`이 **'안'** 이면서 `event_type`이 **`view`** 인 줄의 개수 |
| `prior_uniq_product_cnt` | 그 줄들의 `product_id`에서 중복을 뺀 개수 |
| `cart_product_view_cnt` | 그 줄들 중 `product_id`가 **담은 상품 번호와 같은** 것의 개수 |
| `has_prior_view` | 위 조회 수가 0이면 0, 1 이상이면 1 |
| `purchase_7d` | `7일창`이 **'안'** 인 줄이 하나라도 있으면 1, 없으면 0 |

> **`← t0` 로 표시된 `cart` 줄은 세지 않는다.** 담은 행위 자체는 "담기 전 조회"가 아니다.
> `30분창`이 **'밖'** 인 줄, `7일창`이 **'t0 이전'** 이나 **'밖'** 인 줄도 세지 않는다.
> 그 줄들은 **경계가 제대로 그어졌는지 확인하라고** 일부러 보여주는 것이다.

#### 틀리면 이런 모양이다

```
테이블 → 조회 3건 / 종류 1        '안'인 view 줄의 상품이 두 종류인데 종류가 1   → 불일치
테이블 → ... || Y 0               7일창 '안'인 구매 줄이 있는데 Y가 0            → 불일치
테이블 → 담은상품 2               담은 상품과 번호가 다른 줄을 세고 있다        → 불일치
```

**20명 중 한 명이라도 어긋나면 09번으로 넘어가지 않는다.**

#### 체크리스트

```
[ ] view 줄 개수        = 조회 N건
[ ] 그 줄들의 상품 종류 수 = 종류
[ ] 담은 상품과 같은 줄 수 = 담은상품
[ ] 조회 0이면 has_prior_view 0, 1 이상이면 1
[ ] 7일창 '안'인 구매 줄 유무 = Y
```

In [48]:
log = read_parquet(OUT + r"\01_qtcheck.parquet",
                   columns=["user_id", "event_time", "event_type", "product_id"])

sample_ids = master["user_id"].sample(20, random_state=3217).to_list()
sub = log.loc[log["user_id"].isin(sample_ids)]     # 20명치만 남긴다

del log
import gc; gc.collect()

print(f"대조 대상 {len(sample_ids)}명  |  해당 원본 기록 {len(sub):,}행")

대조 대상 20명  |  해당 원본 기록 425행


In [49]:
SHOW_MIN = 60      # X 근거를 보여줄 범위(분). 규칙은 30분이지만 넓게 보여준다
Y_DAYS   =  7      # Y 규칙(일)


def fmt(df, cols, empty):
    if len(df) == 0:
        return "     " + empty
    return "     " + df[cols].to_string(index=False).replace("\n", "\n     ")


for i, uid in enumerate(sample_ids, 1):
    r   = master.loc[master["user_id"] == uid].iloc[0]
    t0  = r["t0"]
    pid = r["product_id"]
    one = sub.loc[sub["user_id"] == uid]

    print("=" * 94)
    print(f"[{i:>2}] user_id {uid}   t0 {t0}   담은 상품 {pid}   "
          f"price {r['price']}   brand_grp {r['brand_grp']}")
    print(f"     테이블의 주장 →  조회 {r['prior_view_cnt']}건 / 종류 {r['prior_uniq_product_cnt']}"
          f" / 담은상품 {r['cart_product_view_cnt']} / has_prior_view {r['has_prior_view']}"
          f"  ||  Y {r['purchase_7d']}"
          + (f" (lag {r['lag_hours']:.2f}h)" if r["purchase_7d"] == 1 else ""))

    # ── X 근거 : 규칙(30분)보다 넓게 보여주고, 창 판정은 눈으로 한다 ──
    L = one.loc[(one["event_time"] >= t0 - Timedelta(minutes=SHOW_MIN))
                & (one["event_time"] <= t0)].sort_values("event_time").copy()

    gap = (t0 - L["event_time"]).dt.total_seconds() / 60      # 반올림 전 값으로 판정
    L["t0까지(분)"] = gap.round(2)
    L["30분창"]     = "밖"
    L.loc[gap <= WINDOW, "30분창"]                                = "안"
    L.loc[L["event_type"] != "view", "30분창"]                    = "— (view 아님)"
    L.loc[(L["event_time"] == t0) & (L["event_type"] == "cart"),
          "30분창"]                                              = "← t0 (세지 않음)"

    print(f"  X 근거 — t0 직전 {SHOW_MIN}분   ('안' + view 인 줄만 센다)")
    print(fmt(L, ["t0까지(분)", "event_time", "event_type", "product_id", "30분창"],
              "(기록 없음 → 조회·종류·담은상품 전부 0)"))

    # ── Y 근거 : 그 상품 구매 기록 전부. 7일 판정은 눈으로 한다 ──
    R = one.loc[(one["event_type"] == "purchase")
                & (one["product_id"] == pid)].sort_values("event_time").copy()

    hrs = (R["event_time"] - t0).dt.total_seconds() / 3600
    R["t0부터(시간)"] = hrs.round(2)
    R["7일창"] = ["t0 이전" if h <= 0 else ("안" if h <= Y_DAYS * 24 else "밖")
                  for h in hrs]

    print(f"  Y 근거 — 이 고객이 이 상품을 산 기록 전부 (기간 무관)   "
          f"('안'이 하나라도 있으면 Y=1)")
    print(fmt(R, ["t0부터(시간)", "event_time", "product_id", "7일창"],
              "(구매 기록 없음 → Y=0)"))
    print()

[ 1] user_id 250311201   t0 2019-11-03 10:03:59   담은 상품 5671878   price 9.52   brand_grp 미상
     테이블의 주장 →  조회 0건 / 종류 0 / 담은상품 0 / has_prior_view 0  ||  Y 1 (lag 116.50h)
  X 근거 — t0 직전 60분   ('안' + view 인 줄만 센다)
      t0까지(분)          event_time event_type  product_id         30분창
          0.0 2019-11-03 10:03:59       cart     5671878 ← t0 (세지 않음)
  Y 근거 — 이 고객이 이 상품을 산 기록 전부 (기간 무관)   ('안'이 하나라도 있으면 Y=1)
      t0부터(시간)          event_time  product_id 7일창
         116.5 2019-11-08 06:34:12     5671878   안

[ 2] user_id 565763975   t0 2019-10-30 15:34:21   담은 상품 5799891   price 19.03   brand_grp 미상
     테이블의 주장 →  조회 1건 / 종류 1 / 담은상품 1 / has_prior_view 1  ||  Y 0
  X 근거 — t0 직전 60분   ('안' + view 인 줄만 센다)
      t0까지(분)          event_time event_type  product_id         30분창
         1.27 2019-10-30 15:33:05       view     5799891            안
         0.00 2019-10-30 15:34:21       cart     5799891 ← t0 (세지 않음)
  Y 근거 — 이 고객이 이 상품을 산 기록 전부 (기간 무관)   ('안'이 하나라도 있으면 Y=1)
     (구매 기록 없음

## #07. 저장

- 09번 EDA부터 이 파일 하나만 쓴다.
- 108,458행이라 CSV로 둔다. 엑셀로 열어 눈으로 확인할 수 있다(교재 LAB-03 5).
- `utf-8-sig`로 저장해 엑셀에서 한글이 깨지지 않게 한다.

In [50]:
PATH = OUT + r"\03_master.csv"

master.to_csv(PATH, index=False, encoding="utf-8-sig")

print("저장 완료 :", PATH)
print(f"{len(master):,}행 × {master.shape[1]}열")
print()
master.info()

저장 완료 : .\output\03_master.csv
108,458행 × 16열

<class 'pandas.DataFrame'>
RangeIndex: 108458 entries, 0 to 108457
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   user_id                 108458 non-null  int64         
 1   t0                      108458 non-null  datetime64[us]
 2   product_id              108458 non-null  int32         
 3   purchase_7d             108458 non-null  int8          
 4   lag_hours               22236 non-null   float64       
 5   price                   108458 non-null  float64       
 6   prior_view_cnt          108458 non-null  int32         
 7   prior_uniq_product_cnt  108458 non-null  int32         
 8   cart_product_view_cnt   108458 non-null  int32         
 9   has_prior_view          108458 non-null  int8          
 10  cart_timeband           108458 non-null  category      
 11  cart_weekday            108458 non-null  category      

## #08. Y 결과 대조 (수기 대조 후 보조 검사)

In [51]:
import os, sys

os.chdir(BASE)
sys.path.insert(0, BASE)

from pandas import read_csv, read_parquet, to_datetime, Timedelta

OUT  = BASE + r"\output"


In [52]:
m = read_csv(OUT + r"\03_master.csv", encoding="utf-8-sig")
m["t0"]         = to_datetime(m["t0"], format="%Y-%m-%d %H:%M:%S")
m["product_id"] = m["product_id"].astype("int32")

log = read_parquet(OUT + r"\01_qtcheck.parquet",
                   columns=["user_id", "product_id", "event_time", "event_type"])
buy = log.loc[log["event_type"] == "purchase", ["user_id", "product_id", "event_time"]]

del log
import gc; gc.collect()

print(f"마스터 {len(m):,}행  |  구매 이벤트 {len(buy):,}행")


마스터 108,458행  |  구매 이벤트 567,514행


In [53]:
key = m[["user_id", "product_id", "t0"]]

b = buy.merge(key, on=["user_id", "product_id"], how="inner")   # 담은 그 상품의 구매만
b["lag"] = (b["event_time"] - b["t0"]) / Timedelta(hours=1)

hit = b.loc[(b["lag"] > 0) & (b["lag"] <= 168)]                 # t0 제외, 168시간 포함
chk = hit.groupby("user_id")["lag"].min().rename("lag_chk")

v = m[["user_id", "purchase_7d", "lag_hours"]].merge(chk, on="user_id", how="left")
v["y_chk"] = v["lag_chk"].notna().astype("int8")

print(f"재계산 Y=1 : {int(v['y_chk'].sum()):,}명")
print(f"테이블 Y=1 : {int(v['purchase_7d'].sum()):,}명")


재계산 Y=1 : 22,236명
테이블 Y=1 : 22,236명


In [54]:
fp = int(((v["purchase_7d"] == 1) & (v["y_chk"] == 0)).sum())
fn = int(((v["purchase_7d"] == 0) & (v["y_chk"] == 1)).sum())

both     = v.loc[(v["purchase_7d"] == 1) & (v["y_chk"] == 1)]
bad_lag  = int(((both["lag_hours"] - both["lag_chk"]).abs() > 1e-4).sum())
leak     = int(v.loc[v["purchase_7d"] == 0, "lag_hours"].notna().sum())

print(f"Y=1 인데 창 안 구매 없음 : {fp:,}   (0이어야 함)")
print(f"Y=0 인데 창 안 구매 있음 : {fn:,}   (0이어야 함)")
print(f"lag_hours 불일치         : {bad_lag:,}   (0이어야 함)")
print(f"Y=0 인데 lag 값 있음     : {leak:,}   (0이어야 함)")
print()
print("전부 통과 :", fp == 0 and fn == 0 and bad_lag == 0 and leak == 0)


Y=1 인데 창 안 구매 없음 : 0   (0이어야 함)
Y=0 인데 창 안 구매 있음 : 0   (0이어야 함)
lag_hours 불일치         : 0   (0이어야 함)
Y=0 인데 lag 값 있음     : 0   (0이어야 함)

전부 통과 : True


In [55]:
L = m.loc[m["purchase_7d"] == 1, "lag_hours"]

print(f"lag 최소 {L.min():.4f} h  /  최대 {L.max():.4f} h")
print(f"0 이하   : {int((L <= 0).sum())}   (0이어야 함 — 담기 전 구매는 제외되었어야)")
print(f"168 초과 : {int((L > 168).sum())}   (0이어야 함)")
print(f"정확히 168.0 : {int((L == 168).sum())}   (있어도 정상 — 상한 포함)")


lag 최소 0.0039 h  /  최대 167.9747 h
0 이하   : 0   (0이어야 함 — 담기 전 구매는 제외되었어야)
168 초과 : 0   (0이어야 함)
정확히 168.0 : 0   (있어도 정상 — 상한 포함)


## #09. Y=0 고객의 7일 내 "다른 상품" 구매 비율

Y=0은 **담은 그 상품을** 7일 안에 사지 않았다는 뜻이지, 아무것도 사지 않았다는 뜻이 아니다.
Y=0 중 상당수가 다른 상품을 샀다면 "전환 실패"라는 표현 자체가 부정확해진다.

**정의는 산출 전에 확정했다** (브리핑 §6 결정 7번). 결과를 보고 바꾸지 않는다.

| 항목 | 정의 |
|---|---|
| 분모 | `purchase_7d == 0` 고객 86,222명 |
| 분자 | `t₀ < t ≤ t₀+168h` 에 `purchase` 이벤트가 1건 이상 있는 고객 (`product_id` 무관) |
| 경계 | Y와 동일 — 왼쪽 초과(t₀ 제외), 오른쪽 이하(168시간 포함) |
| 성격 | 보고서 서술용. 이 수치로 X·Y 설계를 되돌리지 않는다 |

**"다른 상품" 조건을 따로 걸지 않는다.** Y=0이면 담은 그 상품의 창 안 구매가 정의상 0건이므로
"다른 상품 구매"와 "임의 상품 구매"가 동치다. 대신 이것을 **검산으로 쓴다** —
조건을 건 값과 걸지 않은 값이 다르면 Y 계산에 버그가 있다는 뜻이다.
`#08` 전수 조사에 이어지는 네 번째 검증이 된다.

In [56]:
# ── #09-1. 불러오기 ──────────────────────────────────────────────────────
# #08을 방금 돌렸다면 m·buy가 이미 메모리에 있지만, 이 셀을 다시 돌려도 문제 없다.
# 셀 하나만 따로 실행하는 경우를 대비해 자체 완결로 둔다.

import os, sys, gc

os.chdir(BASE)                          # helpers 폰트 경로가 상대경로라 필요하다
sys.path.insert(0, BASE)

from pandas import read_csv, read_parquet, to_datetime, Timedelta

OUT  = BASE + r"\output"

Y_HOURS = 168      # 7일. Y(purchase_7d)와 같은 값을 쓴다. 경계 규칙도 같다

# ── 마스터 테이블 ──
# CSV는 타입을 보존하지 않는다. 시차를 재야 하므로 t0를 datetime으로 되살린다.
m = read_csv(OUT + r"\03_master.csv", encoding="utf-8-sig")
m["t0"]         = to_datetime(m["t0"], format="%Y-%m-%d %H:%M:%S")
m["product_id"] = m["product_id"].astype("int32")

# ── 원본 로그에서 구매 이벤트만 ──
# 필요한 4열만 읽는다. 9열 전부 읽으면 메모리가 두 배 이상 든다.
log = read_parquet(OUT + r"\01_qtcheck.parquet",
                   columns=["user_id", "product_id", "event_time", "event_type"])

buy = log.loc[log["event_type"] == "purchase",
              ["user_id", "product_id", "event_time"]].copy()

del log            # 8,259,159행짜리는 바로 버린다
gc.collect()

print(f"마스터      : {len(m):,}행")
print(f"구매 이벤트 : {len(buy):,}행   (#08에서 본 567,514행과 같아야 한다)")

마스터      : 108,458행
구매 이벤트 : 567,514행   (#08에서 본 567,514행과 같아야 한다)


In [57]:
# ── #09-2. 분모 만들기 : Y=0 고객 ────────────────────────────────────────
# 정의상 분모는 purchase_7d == 0 인 고객 '전원'이다. 조건을 더 걸지 않는다.

y0 = m.loc[m["purchase_7d"] == 0, ["user_id", "product_id", "t0"]].copy()

# ⚠️ 이름을 미리 갈라둔다.
#    곧 구매 로그와 붙일 텐데 양쪽 다 product_id 를 갖고 있어서
#    그대로 두면 product_id_x / product_id_y 가 되어 어느 쪽이 무엇인지 헷갈린다.
#    cart_pid = '이 고객이 담았던 상품'
y0 = y0.rename(columns={"product_id": "cart_pid"})

DENOM = len(y0)

print(f"분모 (Y=0 고객) : {DENOM:,}명")
print(f"브리핑 값 86,222명과 일치 : {DENOM == 86222}")
print(f"user_id 중복 : {y0['user_id'].duplicated().sum()}   (1행=1고객이므로 0이어야 한다)")

분모 (Y=0 고객) : 86,222명
브리핑 값 86,222명과 일치 : True
user_id 중복 : 0   (1행=1고객이므로 0이어야 한다)


In [58]:
# ── #09-3. 7일 창 적용 ───────────────────────────────────────────────────
# Y=0 고객이 남긴 구매 기록을 전부 가져와 t0 기준 시차를 잰다.
#
#   · user_id 로만 붙인다 → 상품을 가리지 않는다. 이게 이 산출의 핵심이다.
#     (Y 계산은 user_id + product_id 두 개로 붙였다. 여기가 결정적으로 다른 지점이다)
#   · 구매한 상품 번호는 buy_pid 로 이름을 바꿔 cart_pid 와 구분한다

b = buy.rename(columns={"product_id": "buy_pid"}).merge(y0, on="user_id", how="inner")

# 시차(시간 단위). 음수면 t0 이전 구매, 0이면 같은 초.
b["lag"] = (b["event_time"] - b["t0"]) / Timedelta(hours=1)

# 경계 규칙은 Y와 완전히 같다 — 왼쪽 초과(t0 제외), 오른쪽 이하(168시간 포함)
hit = b.loc[(b["lag"] > 0) & (b["lag"] <= Y_HOURS)]

print(f"Y=0 고객의 구매 기록 (기간 무관) : {len(b):,}건")
print(f"  · t0 이전 또는 같은 초         : {int((b['lag'] <= 0).sum()):,}건   (창 밖)")
print(f"  · 7일 창 안                    : {len(hit):,}건   ← 이것만 센다")
print(f"  · 7일 초과                     : {int((b['lag'] > Y_HOURS).sum()):,}건   (창 밖)")

Y=0 고객의 구매 기록 (기간 무관) : 137,861건
  · t0 이전 또는 같은 초         : 654건   (창 밖)
  · 7일 창 안                    : 62,046건   ← 이것만 센다
  · 7일 초과                     : 75,161건   (창 밖)


In [59]:
# ── #09-4. 산출과 검산 ───────────────────────────────────────────────────
# A : 정의 그대로. 상품을 가리지 않고 '창 안에 뭐라도 산' 고객 수
# B : 담은 그 상품을 뺀 것. 정의상 A와 같아야 한다.
#
#     왜 같아야 하나 — Y=0 이라는 말 자체가 "담은 그 상품을 창 안에 사지 않았다"는 뜻이다.
#     그러므로 창 안 구매 기록에 담은 상품이 섞여 있을 수가 없다.
#     A != B 가 나오면 이 산출이 틀린 게 아니라 Y 계산이 틀린 것이다.

nA = hit["user_id"].nunique()
nB = hit.loc[hit["buy_pid"] != hit["cart_pid"], "user_id"].nunique()

same_pid = int((hit["buy_pid"] == hit["cart_pid"]).sum())   # 0이어야 한다

print(f"A. 창 안에 무엇이든 산 고객 : {nA:,}명")
print(f"B. 창 안에 다른 상품을 산 고객 : {nB:,}명")
print()
print(f"검산 ① A == B                 : {nA == nB}")
print(f"검산 ② 담은 상품이 섞인 건수  : {same_pid:,}   (0이어야 한다)")
print(f"검산 통과 : {nA == nB and same_pid == 0}")
print()

rate = nA / DENOM * 100

print("─" * 60)
print(f"  결정 7번 결과")
print(f"  Y=0 고객 {DENOM:,}명 중 {nA:,}명이 7일 내 다른 상품을 구매 = {rate:.2f}%")
print("─" * 60)

A. 창 안에 무엇이든 산 고객 : 6,460명
B. 창 안에 다른 상품을 산 고객 : 6,460명

검산 ① A == B                 : True
검산 ② 담은 상품이 섞인 건수  : 0   (0이어야 한다)
검산 통과 : True

────────────────────────────────────────────────────────────
  결정 7번 결과
  Y=0 고객 86,222명 중 6,460명이 7일 내 다른 상품을 구매 = 7.49%
────────────────────────────────────────────────────────────


In [62]:
# ── #09-5. 산출값 정리 ───────────────────────────────────────────────────
# ⚠️ 이 셀은 숫자만 낸다. 해석 문장은 코드가 쓰지 않는다.
#    앞 버전은 결과와 무관하게 결론 문장 하나를 고정 출력했는데,
#    그건 사전 확정이 아니라 결론을 미리 박아둔 것이라 폐기했다.

n_any  = int(m["purchase_7d"].sum())        # 담은 그 상품 구매 (Y=1)
n_else = nA                                 # 그 상품은 아니고 다른 걸 구매
n_none = DENOM - nA                          # 아무것도 구매 안 함
N      = len(m)

print(f"{'7일 동안':<28}{'인원':>10}{'전체 대비':>12}")
print("─" * 50)
print(f"{'담은 그 상품을 샀다 (Y=1)':<24}{n_any:>10,}{n_any/N*100:>11.2f}%")
print(f"{'다른 상품만 샀다':<26}{n_else:>10,}{n_else/N*100:>11.2f}%")
print(f"{'아무것도 안 샀다':<26}{n_none:>10,}{n_none/N*100:>11.2f}%")
print("─" * 50)
print(f"{'합계':<26}{n_any+n_else+n_none:>10,}{(n_any+n_else+n_none)/N*100:>11.2f}%")
print()
print(f"Y=0 기준 : {DENOM:,}명 중 {nA:,}명({nA/DENOM*100:.2f}%)이 다른 상품 구매,")
print(f"           {n_none:,}명({n_none/DENOM*100:.2f}%)은 아무것도 구매하지 않음")
print()
print("→ 사전에 정한 서술 방향(브리핑 §6 결정 7번) 두 갈래를 보고,")
print("   어느 쪽인지 사람이 판단해 한계 문장을 직접 쓴다.")

7일 동안                               인원       전체 대비
──────────────────────────────────────────────────
담은 그 상품을 샀다 (Y=1)           22,236      20.50%
다른 상품만 샀다                      6,460       5.96%
아무것도 안 샀다                     79,762      73.54%
──────────────────────────────────────────────────
합계                           108,458     100.00%

Y=0 기준 : 86,222명 중 6,460명(7.49%)이 다른 상품 구매,
           79,762명(92.51%)은 아무것도 구매하지 않음

→ 사전에 정한 서술 방향(브리핑 §6 결정 7번) 두 갈래를 보고,
   어느 쪽인지 사람이 판단해 한계 문장을 직접 쓴다.
